# VAE training and processing

Code to train a new VAE and run the CSI processing.

## Setup

In [ ]:
import csv
import zipfile
from pathlib import Path
from string import ascii_uppercase
from urllib.request import urlretrieve

import numpy as np
import scipy.io as sio
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

# Dataset config
DATASET_URL = "https://zenodo.org/record/7732595/files/S1.zip"
"""Zenodo URL for the S1 dataset."""
DATASET_PATH = Path("dataset/S1")
"""Local path to store the S1 dataset."""
N_ACTIVITIES = 12
"""Total number of activities in the dataset"""
N_SAMPLES = 12000
"""Number of samples to extract from each CSI matrix file."""
WINDOW_SIZE = 450
"""Size of the sliding window to extract from each sample."""
N_ANTENNAS = 1
"""Total number of antennas used, either a single one or all of them."""
ANTENNA = 0
"""If N_ANTENNAS==1, select which antenna to use (0 to 3). Otherwise, this value is ignored."""

# Categorical VAE config
LATENT_DIM = 10
CATEGORICAL_DIM = N_ACTIVITIES
VAE_NAME = f"vaed_s1a_a{ANTENNA}_ls{LATENT_DIM}" if N_ANTENNAS == 1 else f"vaed_s1a_f_ls{LATENT_DIM}"
CHECKPOINT_DIR = Path(f"./VAED_models_{N_ACTIVITIES}activities/{VAE_NAME}")

# Training config
BATCH_SIZE = 25
NUM_EPOCHS = 1
PATIENCE = 3
LEARNING_RATE = 1e-3
USE_GPU = torch.cuda.is_available()
MODEL_DEVICE = torch.device("cuda" if USE_GPU else "cpu")

### Download the dataset

In [ ]:
if not DATASET_PATH.exists() or not (DATASET_PATH / "S1").exists():
    DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)

    zip_file_path = DATASET_PATH.with_suffix(".zip")

    urlretrieve(DATASET_URL, zip_file_path)

    with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
        zip_ref.extractall("dataset")

    zip_file_path.unlink()

## CSI dataset loading

In [ ]:
class CSIDataset(Dataset):
    """CSI Dataset for PyTorch.

    Shape of dataset items is (n_antennas, window_size, n_subcarriers)
    """

    def __init__(
        self,
        files: list[Path],
        n_samples: int,
        window_size: int,
        n_antennas: int,
        antenna_select: int,
        normalize: bool = True,
    ):
        self.window_size = window_size
        self.n_antennas = n_antennas
        self.normalize = normalize

        self.data = []
        self.labels = []
        self.index_map = []

        global_max = 0.0

        # Load files once, build index map
        for label, file in enumerate(files):
            # num_samples, n_subcarriers, n_antennas
            mat = sio.loadmat(file)

            csi = np.array(mat["csi"])
            csi = csi[:n_samples, ..., int(antenna_select)] if n_antennas == 1 else csi[:n_samples]
            csi = np.round(np.abs(csi)).astype(np.float32)

            if normalize:
                global_max = max(global_max, csi.max())

            file_id = len(self.data)
            self.data.append(csi)
            self.labels.append(label)

            # Build lazy sliding-window index
            for start in range(n_samples - window_size):
                self.index_map.append((file_id, start))

        self.global_max = global_max if normalize else 1.0

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx: int):
        file_id, start = self.index_map[idx]
        csi = self.data[file_id]

        window = csi[start : start + self.window_size]

        # (window_size, n_subcarriers, n_antennas) → (n_antennas, window_size, n_subcarriers)
        window = window[np.newaxis, ...] if self.n_antennas == 1 else np.transpose(window, (2, 0, 1))

        x = torch.from_numpy(window) / self.global_max
        y = self.labels[file_id]

        return x, y

In [ ]:
files = [DATASET_PATH / f"S1a_{x}.mat" for x in ascii_uppercase[:N_ACTIVITIES]][:2]
print([f.name for f in files])

# Shape of dataset samples: (n_antennas, window_size, n_subcarriers)
dataset = CSIDataset(
    files=files,
    n_samples=N_SAMPLES,
    window_size=WINDOW_SIZE,
    n_antennas=N_ANTENNAS,
    antenna_select=ANTENNA,
)

# Shape of dataloader batches: (batch_size, n_antennas, window_size, n_subcarriers)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4 if USE_GPU else 0,
    pin_memory=USE_GPU,
)

['S1a_A.mat', 'S1a_B.mat']


## Variational Auto-Encoder

In [ ]:
def sample_gumbel(shape: torch.Size, eps: float = 1e-20) -> torch.Tensor:
    """Sample Gumbel noise.

    Args:
        shape (torch.Size): Shape of the output tensor.
        eps (float): Small constant for numerical stability of logarithms.

    Returns:
        torch.Tensor: Sampled Gumbel noise.

    """
    # Sample from a uniform distribution
    unif = torch.rand(shape)

    # Convert uniform samples to Gumbel samples
    return -torch.log(-torch.log(unif + eps) + eps)

class GumbelSoftmaxSampling(nn.Module):
    """Gumbel-Softmax sampling module."""

    def forward(self, logits: torch.Tensor, temperature: float) -> torch.Tensor:
        """Draw a sample from the Gumbel-Softmax distribution.

        Args:
            logits (torch.Tensor): Logits of the categorical distribution.
            temperature (float): Temperature parameter.

        Returns:
            torch.Tensor: Sampled tensor from the Gumbel-Softmax distribution.

        """
        # Sum log probabilities and Gumbel noise
        y = logits + sample_gumbel(logits.size())

        # Use softmax for differentiability
        y_max = nn.functional.softmax(y / temperature, dim=-1)

        return y_max.view(-1, LATENT_DIM, CATEGORICAL_DIM)

In [ ]:
class CSIEncoder(nn.Module):
    """CSI encoder module for VAE."""

    def __init__(self, input_shape: tuple[int, int, int], latent_dim: int, categorical_dim: int):
        super().__init__()

        self.latent_dim = latent_dim
        self.categorical_dim = categorical_dim

        self.conv = nn.Sequential(
            nn.Conv2d(input_shape[2], 32, kernel_size=(5, 8), stride=(5, 8)),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=(5, 8), stride=(5, 8)),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=(2, 4), stride=(2, 4)),
            nn.ReLU(),
        )

        # Infer flattened size dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, input_shape[2], input_shape[0], input_shape[1])
            flat_dim = self.conv(dummy).view(1, -1).size(1)

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 16),
            nn.ReLU(),
        )

        self.sampling = GumbelSoftmaxSampling()

    def forward(self, x: torch.Tensor, temperature: float) -> torch.Tensor:
        """Forward pass through the encoder."""
        x = self.conv(x)
        x = self.fc(x)

        x = x.view(-1, self.latent_dim, self.categorical_dim)
        z = self.sampling(x, temperature)
        return z.view(-1, self.latent_dim * self.categorical_dim)


class CSIDecoder(nn.Module):
    """CSI decoder module for VAE."""

    def __init__(self, input_shape: tuple[int, int, int], latent_dim: int, categorical_dim: int, out_filter: int):
        super().__init__()

        self.input_shape = input_shape
        flat_dim = input_shape[0] * input_shape[1] * input_shape[2]

        self.fc = nn.Sequential(
            nn.Linear(latent_dim * categorical_dim, flat_dim),
            nn.ReLU(),
        )

        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(input_shape[2], 32, kernel_size=(2, 4), stride=(2, 4), padding=0),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 32, kernel_size=(5, 8), stride=(5, 8), padding=2),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 32, kernel_size=(5, 8), stride=(5, 8), padding=2),
            nn.ReLU(),
            nn.ConvTranspose2d(32, out_filter, kernel_size=out_filter, padding=out_filter // 2),
            nn.Sigmoid(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """Forward pass through the decoder."""
        x = self.fc(z)
        x = x.view(
            z.size(0),
            self.input_shape[2],
            self.input_shape[0],
            self.input_shape[1],
        )
        return self.deconv(x)

In [ ]:
class VAE(nn.Module):
    """Variational Autoencoder for CSI data."""

    def __init_(
        self,
        enc_input_shape: tuple[int, int, int] = (450, 2048, 1),
        dec_input_shape: tuple[int, int, int] = (9, 8, 32),
        latent_dim: int = 2,
        categorical_dim: int = 2,
    ) -> None:
        super().__init__()

        self.encoder = CSIEncoder(enc_input_shape, latent_dim, categorical_dim)
        self.decoder = CSIDecoder(dec_input_shape, latent_dim, categorical_dim, enc_input_shape[-1])

    def forward(self, x: torch.Tensor, temperature: float) -> torch.Tensor:
        """Forward pass through the VAE."""
        z = self.encoder(x, temperature)
        return self.decoder(z)


def vae_loss(
    x: torch.Tensor,
    recon: torch.Tensor,
    z_mean: torch.Tensor,
    z_log_var: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute the VAE loss as the sum of reconstruction loss and KL divergence."""
    # Reconstruction loss
    recon_loss = nn.functional.binary_cross_entropy(recon, x, reduction="none").sum(dim=(1, 2, 3)).mean()

    # KL divergence
    kl_loss = -0.5 * (1 + z_log_var - z_mean.pow(2) - z_log_var.exp())
    kl_loss = kl_loss.sum(dim=1).mean()

    return recon_loss + kl_loss, recon_loss, kl_loss

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def save_checkpoint(model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int) -> None:
    """Save model and optimizer state dicts as a checkpoint."""
    path = CHECKPOINT_DIR / f"cp-{epoch:04d}.pt"
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        },
        path,
    )


class EarlyStopping:
    """Early stopping utility to halt training when validation loss stops improving."""

    def __init__(self, patience: int) -> None:
        self.patience = patience
        self.best_loss = float("inf")
        self.counter = 0

    def step(self, loss: float) -> bool:
        """Check if training should stop early based on loss improvement."""
        # If loss improved, reset counter and continue training
        if loss < self.best_loss:
            self.best_loss = loss
            self.counter = 0
            return False

        # If loss did not improve, increment counter
        self.counter += 1
        return self.counter >= self.patience


log_path = f"{CHECKPOINT_DIR}/model_history_log.csv"
file_exists = Path(log_path).is_file()

with Path(log_path).open("a", newline="") as csv_file:
    csv_writer = csv.writer(csv_file)

if not file_exists:
    csv_writer.writerow(["epoch", "loss", "reconstruction_loss", "kl_loss"])

In [ ]:
model = VAE(latent_dim=LATENT_DIM, categorical_dim=CATEGORICAL_DIM).to(MODEL_DEVICE)
model.train()

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
early_stopping = EarlyStopping(patience=PATIENCE)

for epoch in range(NUM_EPOCHS):
    model.train()

    epoch_loss = 0.0
    epoch_recon = 0.0
    epoch_kl = 0.0

    for x, _ in dataloader:
        x_dev = x.to(MODEL_DEVICE)

        optimizer.zero_grad()
        recon, z_mean, z_log_var = model(x_dev)
        loss, recon_loss, kl_loss = vae_loss(x_dev, recon, z_mean, z_log_var)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_recon += recon_loss.item()
        epoch_kl += kl_loss.item()

    epoch_loss /= len(dataloader)
    epoch_recon /= len(dataloader)
    epoch_kl /= len(dataloader)

    save_checkpoint(model, optimizer, epoch)
    csv_writer.writerow([epoch, epoch_loss, epoch_recon, epoch_kl])
    csv_file.flush()

    if early_stopping.step(epoch_loss):
        break